In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ \text{Patient: } [\text{Temp}: 38.2 \degree C, \text{Rhinitis}: 1, \text{Cought}: 0, \text{Sore Throat}: 1, \text{Headache}: 1] \xrightarrow{} \text{Disease} $$
$$ \text{Patient: } [\text{Temp}: 36.5 \degree C, \text{Rhinitis}: 0, \text{Cought}: 0, \text{Sore Throat}: 0, \text{Headache}: 1] \xrightarrow{} \text{Healthy} $$
$$ \text{Patient: } [\text{Temp}: 37.5 \degree C, \text{Rhinitis}: 0, \text{Cought}: 1, \text{Sore Throat}: 0, \text{Headache}: 1] \xrightarrow{} \text{???} $$

In [ ]:
from torch.nn import Linear, Sequential, Sigmoid
from torch.nn.functional import binary_cross_entropy, binary_cross_entropy_with_logits
from torch.optim import SGD


import import_ipynb
from common import assert_eq, assert_ge, assert_lt, Patient, T # type: ignore


# Convert Patient from 3-classes {healthy, viral, bacterial} to 2-classes {healthy, sick}
def Patient2C(healthy: float) -> Patient:
    return Patient([healthy, 1 - healthy, 0.00])


def logistic_regression_2c_sgd_autograd(X, y, epochs=2000, lr=0.1) -> tuple[float, callable]:
    """
    Perform logistic regression using Stochastic Gradient Descent (SGD) with autograd.

    Args:
        X: Input features of shape (Samples, Features)
        y: Target values of shape (Samples, 1)
        epochs: Number of training epochs
        lr: Learning rate

    Returns:
        A tuple containing the final loss and a prediction function that takes new input data and returns predicted probabilities.
    """

    (s, f) = X.shape

    # `BCEWithLogitsLoss` combines a `Sigmoid` layer and the `BCELoss` in one single class. 
    # This version is more numerically stable than using a plain `Sigmoid` followed by 
    # a `BCELoss` as, by combining the operations into one layer, we take advantage of 
    # the log-sum-exp trick for numerical stability.
    model = Linear(f, 1)
    loss_fn = binary_cross_entropy_with_logits

    # model = Sequential(Linear(f, 1), Sigmoid())
    # loss_fn = binary_cross_entropy

    optimizer = SGD(model.parameters(), lr=lr)

    for _ in range(epochs):
        optimizer.zero_grad()

        logits = model(X)
        assert_eq(logits.shape, (s, 1))

        loss = loss_fn(logits, y)
        assert_eq(loss.shape, ())
        
        loss.backward()
        optimizer.step()

    return (loss.item(), model)


def _test_logistic_regression_2c_sgd_autograd(epochs=2000, lr=0.1) -> None:
    training_data = T([Patient2C(0.5).data for _ in range(80)])

    X = training_data[:, :-1]
    X[:, 0] /= 100 # Temperature scaling
    X[:, 5] /= 200 # CRP scaling    
    y = training_data[:, -1].unsqueeze(1)

    (_, model) = logistic_regression_2c_sgd_autograd(X, y, epochs, lr)

    for d in T([Patient2C(0.0).data for _ in range(10)]):
        d[0] /= 100 # The same data scaling as during training.
        d[5] /= 200 # The same data scaling as during training.
        assert_ge(model(d[:-1]), 0.5)
        
    for d in T([Patient2C(1.0).data for _ in range(10)]):
        d[0] /= 100 # The same data scaling as during training.
        d[5] /= 200 # The same data scaling as during training.
        assert_lt(model(d[:-1]), 0.5)
    

if __name__ == "__main__":
    _test_logistic_regression_2c_sgd_autograd()